# 02 — Read Source Blob via SAS Token
**Day 1 | Part 7.4**

Reads pre-loaded source data from the shared external Azure Blob Storage
(`dataenggdailystorage`) using a SAS token stored in **your** Key Vault.

---

## What is this notebook doing?

The project has **two different storage accounts**:

| Storage Account | Owner | What it holds | How you connect |
|---|---|---|---|
| `evdatalakedev` | You | Your Bronze / Silver / Gold data | SP OAuth (`abfss://`) |
| `dataenggdailystorage` | Course instructor | Raw source CSV files | SAS Token (`wasbs://`) |

This notebook connects to the **instructor's** storage account and reads source files from it.
You have **read + list only** access — you cannot write or delete anything there.

---

## Why `wasbs://` and not `abfss://`?

- `abfss://` = **Azure Data Lake Storage Gen2** protocol. Requires OAuth (Service Principal login). Only works when the storage account has **Hierarchical Namespace** enabled AND you have been given an RBAC role on it.
- `wasbs://` = **Windows Azure Storage Blob Service** protocol. Works with a SAS token. Simpler, no OAuth needed. Works on any Azure Blob Storage account.

Since `dataenggdailystorage` belongs to someone else and they gave you a SAS token (not an SP), you use `wasbs://`.

---

## Secrets to add in your Key Vault before running

| Secret Name | Value |
|---|---|
| `source-storage-account` | `dataenggdailystorage` |
| `source-container` | `source` |
| `source-sas-token` | Provided during the session |

## Cell 1 — Load credentials from Key Vault + configure Spark

**What this cell does:**
1. Reads 3 secrets from Key Vault — nothing is hardcoded in the notebook
2. Tells Spark: "when you see a path for `dataenggdailystorage`, use this SAS token to authenticate"

**Line-by-line explanation follows in the code comments.**

In [ ]:
# SCOPE = the name of the Databricks secret scope that is linked to your Key Vault.
# Think of it as a named bridge: "kv-ev-scope" → your Key Vault → the actual secret value.
SCOPE = "kv-ev-scope"

try:
    # dbutils.secrets.get() reads a secret from Key Vault WITHOUT ever showing the value.
    # Even if you print it, Databricks shows [REDACTED] — the value never appears in logs.
    # key= is the name of the secret exactly as you created it in Key Vault.
    STORAGE_ACCOUNT = dbutils.secrets.get(scope=SCOPE, key="source-storage-account")
    # STORAGE_ACCOUNT is now "dataenggdailystorage" (but we never see it printed in full)

    CONTAINER = dbutils.secrets.get(scope=SCOPE, key="source-container")
    # CONTAINER is now "source" — the name of the blob container inside that storage account

    SAS_TOKEN = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")
    # SAS_TOKEN is now the long token string like: se=2027-07-30&sp=rl&spr=https&sv=...&sig=...
    # sp=rl means permissions: r=read, l=list — you can list folders and read files, nothing else

    print(f"Storage account : {STORAGE_ACCOUNT}")
    print(f"Container       : {CONTAINER}")
    print(f"SAS token       : [REDACTED]")
    print("Credentials loaded from Key Vault — OK")

except Exception as e:
    print(f"ERROR: {e}")
    print("\nSecrets missing. Add these 3 secrets to Key Vault first:")
    print("  Portal → Key vaults → kv-ev-intelligence-dev → Secrets → + Generate/Import")
    print("  source-storage-account  →  dataenggdailystorage")
    print("  source-container        →  source")
    print("  source-sas-token        →  <value provided during session>")
    raise

# spark.conf.set() sets a configuration key-value pair for this Spark session only.
# It is NOT permanent — cleared when the cluster restarts.
#
# Key format explained:
#   fs.azure.sas          = "use SAS token auth for Azure blob storage"
#   .{CONTAINER}          = which container inside the storage account
#   .{STORAGE_ACCOUNT}    = which storage account
#   .blob.core.windows.net = the Azure blob endpoint domain
#
# This tells Spark: "whenever you access wasbs://source@dataenggdailystorage.blob.core.windows.net/...,
# authenticate using this SAS token."
spark.conf.set(
    f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SAS_TOKEN
)
print("Spark SAS config set — OK")

## Cell 2 — List top-level folders in the source container

**What this cell does:** Lists everything at the root of the `source` container — like running `ls` on a folder.

**Key terms:**
- `dbutils.fs.ls(path)` — Databricks file system utility. Lists files and folders at a given path. Returns a list of `FileInfo` objects, each with `.name`, `.path`, `.size`.
- `wasbs://` — the protocol prefix telling Spark this is a classic Azure Blob Storage path using SAS token auth.
- `wasbs://{container}@{account}.blob.core.windows.net/` — the full root URL of the container.

In [ ]:
# Build the root URL of the container.
# wasbs://   = protocol (Windows Azure Storage Blob Service, works with SAS token)
# {CONTAINER}@ = the container name ("source") — like a top-level folder in the account
# {STORAGE_ACCOUNT}.blob.core.windows.net = the storage account's domain on Azure
# The trailing / means: start from the root of the container
base_path = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/"

print(f"Listing root of container: {base_path}\n")

try:
    # dbutils.fs.ls() = Databricks utility to list files/folders at a path.
    # Same concept as `ls` in Linux or `dir` in Windows.
    # Returns a list of FileInfo objects — each has: .name, .path, .size, .modificationTime
    items = dbutils.fs.ls(base_path)

    for item in items:
        # item.size == 0 usually means it is a folder (directories have no size)
        item_type = "DIR " if item.size == 0 else "FILE"
        # item.name = just the folder/file name (e.g. "realtime/")
        # item.size = size in bytes (0 for folders)
        print(f"  [{item_type}] {item.name:<50} {item.size:>10} bytes")

    print(f"\n  Total: {len(items)} items found at root level")

except Exception as e:
    print(f"ERROR: {e}")
    print("\nCommon causes:")
    print("  403 Forbidden   → SAS token is wrong, expired, or missing 'l' (list) permission")
    print("                    Check: sp=rl should be in your SAS token string")
    print("  Auth error      → Spark SAS config not set — re-run Cell 1 first")
    raise

## Cell 3 — Drill into the charging_sessions folder

**What this cell does:** Goes one level deeper into `realtime/charging_sessions/` to see how files are organised (e.g. by year/month/day/hour).

This matters because Spark needs to know the folder depth to build the correct glob pattern in Cell 5.

In [ ]:
# Full path to the charging_sessions folder inside the container.
# Structure: wasbs://container@account.blob.core.windows.net/folder/subfolder/
sessions_root = (
    f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"
    f"/realtime/charging_sessions/"
)

print(f"Drilling into: {sessions_root}\n")

try:
    # List the first level — will show year folders like: 2026/
    items = dbutils.fs.ls(sessions_root)
    print("Level 1 — contents of charging_sessions/:")
    for item in items:
        print(f"  {item.name}")

    # Now go one level deeper for each item — will show month folders like: 06/
    print("\nLevel 2 — drilling one folder deeper:")
    for item in items:
        try:
            sub_items = dbutils.fs.ls(item.path)
            for s in sub_items:
                # item.name = e.g. "2026/"   s.name = e.g. "06/"
                print(f"  {item.name}{s.name}")
        except:
            # If item is a file (not a folder), ls() will fail — skip it silently
            pass

except Exception as e:
    print(f"ERROR: {e}")
    print("→ Folder path may differ. Check Cell 2 output for correct folder names.")
    raise

## Cell 4 — Read one specific CSV file

**What this cell does:** Reads a single CSV file into a Spark DataFrame and shows its schema and first 10 rows.

**Key terms:**
- `spark.read` — entry point to load data into Spark. You chain `.option()` and a format method onto it.
- `.option("header", "true")` — tells Spark the first row of the CSV is column names, not data.
- `.option("inferSchema", "true")` — tells Spark to auto-detect column types (integer, string, date, etc.) by scanning the file. Without this, everything is treated as a string.
- `.csv(path)` — reads the file at `path` as a CSV and returns a DataFrame.
- `df.count()` — counts all rows. This triggers a full scan of the file across Spark workers.
- `df.printSchema()` — prints column names and their detected types in a tree view.
- `display(df.limit(10))` — Databricks-specific: shows the first 10 rows as a nice table in the notebook UI.

> **Adjust the date/hour in the path** based on what Cell 3 showed in your container.

In [ ]:
# Full path to ONE specific CSV file.
# The folder structure is: realtime/charging_sessions/YEAR/MONTH/DAY/HOUR/filename.csv
# Adjust 2026/06/01/06 and the filename based on what Cell 3 showed you.
csv_path = (
    f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"
    f"/realtime/charging_sessions/2026/06/01/06/sessions_20260601_0600.csv"
)

print(f"Reading CSV:\n  {csv_path}\n")

try:
    df = (
        spark.read
        # header=true  → row 1 is column names, not a data row
        .option("header", "true")
        # inferSchema=true → Spark scans the file to detect: is this column a number? a date?
        # If false, every column would be StringType — all values treated as plain text
        .option("inferSchema", "true")
        # .csv() = tell Spark the file format is CSV, then give it the path
        .csv(csv_path)
        # df is now a Spark DataFrame — a distributed table held across cluster workers
    )

    print(f"Row count    : {df.count():,}")      # .count() scans all rows — triggers Spark job
    print(f"Column count : {len(df.columns)}")   # df.columns = list of column name strings
    print(f"Columns      : {df.columns}\n")

    # printSchema() prints the column tree with detected types, e.g.:
    #  |-- session_id: string (nullable = true)
    #  |-- energy_kwh: double (nullable = true)
    df.printSchema()

    # display() is a Databricks helper — shows DataFrame as an interactive table in the UI
    # .limit(10) means: only bring back 10 rows — don't scan millions of rows just to preview
    display(df.limit(10))

except Exception as e:
    print(f"ERROR: {e}")
    print("\nCommon causes:")
    print("  'No such file'  → Date/hour in path does not match actual files. Check Cell 3 output.")
    print("  '403 Forbidden' → SAS token missing Read permission (sp must contain 'r')")
    print("  'abfss error'   → You used abfss:// instead of wasbs:// — change the protocol")
    raise

## Cell 5 — Read ALL CSV files in the folder at once (glob pattern)

**What this cell does:** Instead of reading one file, reads every CSV across all year/month/day/hour subfolders in one shot using a **glob pattern**.

**What is a glob pattern?**
A `*` in a path means "match anything here". So:
- `charging_sessions/*` → match any year folder (2025, 2026, ...)
- `charging_sessions/*/*` → match any year/month combination
- `charging_sessions/*/*/*/*/*.csv` → match any CSV file 4 levels deep

Spark scans all matching files and combines them into one single DataFrame automatically — as if they were one big file.

**Why not just point at the top-level folder?**
Spark gets confused if the first level it sees contains only subfolders (no files). The glob pattern tells it exactly how deep to go before it finds the actual `.csv` files.

In [ ]:
# base = the root URL of the storage account + container (no trailing path yet)
base = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"

# glob_path = full path with wildcards.
# /*/*/*/*  = four levels of wildcard = year / month / day / hour
# /*.csv    = any file ending in .csv at that depth
# So the full pattern reads: "find every .csv file that is exactly 4 folder levels
# inside charging_sessions/"
glob_path = f"{base}/realtime/charging_sessions/*/*/*/*/*.csv"

print(f"Reading all CSVs using glob pattern:")
print(f"  {glob_path}\n")

try:
    df_all = (
        spark.read
        .option("header", "true")       # treat row 1 as column names in every file
        .option("inferSchema", "true")  # auto-detect column types across all files
        .csv(glob_path)
        # Spark reads ALL matching files in parallel across cluster workers,
        # then unions them into df_all as one single DataFrame
    )

    total_rows = df_all.count()   # count() triggers the actual read + union across all files
    print(f"Total rows across all files : {total_rows:,}")
    print(f"Column count                : {len(df_all.columns)}")
    print(f"Columns                     : {df_all.columns}\n")
    display(df_all.limit(10))

except Exception as e:
    print(f"ERROR: {e}")
    print("\nTroubleshooting:")
    print("  1. Run Cell 3 first — check the exact folder depth (year/month/day/hour)")
    print("  2. Count the folder levels between charging_sessions/ and the .csv files")
    print("     3 levels deep → /*/*/*/*.csv")
    print("     4 levels deep → /*/*/*/*/*.csv  (this is the current setting)")
    print("  3. Run Cell 4 with one known file first to confirm basic access is working")
    raise

## Cell 6 — Data quality check

**What this cell does:** Checks every column for nulls and empty strings. Flags any column where more than 10% of values are missing.

**Key terms:**
- `import pyspark.sql.functions as F` — imports Spark's built-in column functions (like `F.col()`, `F.isNull()`). Aliased as `F` so you write `F.col()` instead of `functions.col()`.
- `F.col("column_name")` — refers to a column in the DataFrame by name. Like saying "that column" without actually reading data yet.
- `.isNull()` — returns True/False per row: True if the value in that column is null (missing).
- `.cast("string")` — converts the column value to a string type temporarily, so we can check if it is an empty string `""` even if the column is numeric.
- `.filter(condition)` — keeps only the rows where the condition is True. Similar to SQL `WHERE`.
- `.count()` — counts the rows that passed the filter.

In [ ]:
# Import Spark's SQL functions library.
# F is just a short alias — by convention everyone writes "import as F"
import pyspark.sql.functions as F

print("=== Data Quality Check — Source Blob ===\n")

try:
    # df_all.count() = total number of rows across all files read in Cell 5
    total = df_all.count()
    print(f"Total rows : {total:,}")
    print(f"Columns    : {len(df_all.columns)}\n")

    print("Null / empty value counts per column:")
    print(f"  {'Column':<35} {'Nulls':>8}  {'%':>6}  {'Flag'}")
    print("  " + "-" * 60)

    # Loop through every column name in the DataFrame
    for col_name in df_all.columns:

        # Build a filter condition:
        #   F.col(col_name).isNull()          → value is completely missing (NULL)
        #   F.col(col_name).cast("string")==""→ value exists but is an empty string ""
        # The | means OR — flag the row if EITHER condition is true
        null_count = df_all.filter(
            F.col(col_name).isNull() | (F.col(col_name).cast("string") == "")
        ).count()
        # .count() here = how many rows had null or empty in this column

        # Calculate percentage of bad rows
        pct = (null_count / total * 100) if total > 0 else 0

        # Flag columns where more than 10% of rows are null/empty — worth investigating
        flag = "<-- check this" if pct > 10 else ""
        print(f"  {col_name:<35} {null_count:>8,}  {pct:>5.1f}%  {flag}")

    print("\nData quality check complete.")
    print("Silver layer (Day 7) will handle cleaning these nulls.")

except NameError:
    print("ERROR: df_all not found.")
    print("Run Cell 5 first — this cell requires df_all to exist.")
except Exception as e:
    print(f"ERROR: {e}")

## Cell 7 — Final summary

In [ ]:
print("=" * 55)
print("SOURCE BLOB READ TEST — SUMMARY")
print("=" * 55)
print(f"  Storage account : {STORAGE_ACCOUNT}")
print(f"  Container       : {CONTAINER}")
print(f"  Protocol        : wasbs:// (SAS token auth)")
print(f"  Access          : Read + List only (sp=rl)")
print(f"  Total rows read : {df_all.count():,}")
print(f"  Columns         : {len(df_all.columns)}")
print("=" * 55)
print("SAS token read test PASSED — source blob is accessible.")
print("\nNext: proceed to Day 2.")